In [2]:
print("all ok")

all ok


In [3]:
import os
from git import Repo
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationalRetrievalChain
import os

C:\Users\Manas\AppData\Local\Temp\ipykernel_11524\2412092075.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.generic import GenericLoader


In [3]:
mkdir -p test_repo


A subdirectory or file -p already exists.
Error occurred while processing: -p.
A subdirectory or file test_repo already exists.
Error occurred while processing: test_repo.


In [6]:
repo_path = 'test_repo'
repo= Repo.clone_from("https://github.com/praveenr112/End-to-End-Medical-Chatbot",to_path=repo_path)

GitCommandError: Cmd('git') failed due to: exit code(128)
  cmdline: git clone -v -- https://github.com/praveenr112/End-to-End-Medical-Chatbot test_repo
  stderr: 'fatal: destination path 'test_repo' already exists and is not an empty directory.
'

In [4]:
%pwd

'c:\\Praveen\\KrishNaik\\Realtime-Source-Code-Analyzer\\research'

In [7]:
loader = GenericLoader.from_filesystem(repo_path,
                                       glob="**/*",
                                       suffixes=[".py"],
                                       parser=LanguageParser(language=Language.PYTHON,parser_threshold=500)
                                       )

In [8]:
documents = loader.load()

In [9]:
documents

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask import Flask,render_template,jsonify,request\nfrom src.helper import download_embeddings\nfrom langchain_pinecone import PineconeVectorStore\nfrom langchain_openai import ChatOpenAI\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom dotenv import load_dotenv\nfrom src.prompt import system_prompt\nimport os\nfrom pathlib import Path\n\napp = Flask(__name__)\n\nproject_root = Path(__file__).resolve().parent\nload_dotenv(project_root / ".env")\n\nfor env_name in ("PINECONE_API_KEY", "OPENAI_API_KEY"):\n    if not os.getenv(env_name):\n        raise ValueError(f"{env_name} is missing from the environment or .env file.")\n\nembeddings = download_embeddings()\n\nindex_name = "medical-chatbot"\n\ndocsearch = PineconeVectorStore.from_existi

In [10]:
len(documents)

6

In [11]:
documents[0]

Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask import Flask,render_template,jsonify,request\nfrom src.helper import download_embeddings\nfrom langchain_pinecone import PineconeVectorStore\nfrom langchain_openai import ChatOpenAI\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom dotenv import load_dotenv\nfrom src.prompt import system_prompt\nimport os\nfrom pathlib import Path\n\napp = Flask(__name__)\n\nproject_root = Path(__file__).resolve().parent\nload_dotenv(project_root / ".env")\n\nfor env_name in ("PINECONE_API_KEY", "OPENAI_API_KEY"):\n    if not os.getenv(env_name):\n        raise ValueError(f"{env_name} is missing from the environment or .env file.")\n\nembeddings = download_embeddings()\n\nindex_name = "medical-chatbot"\n\ndocsearch = PineconeVectorStore.from_existin

In [12]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON,
                                                                  chunk_size=500,
                                                                  chunk_overlap = 20
                                                                  )

In [13]:
texts = documents_splitter.split_documents(documents)

In [14]:
len(texts)

13

In [15]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')


In [16]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [17]:
embedddings = OpenAIEmbeddings(disallowed_special=())

In [18]:
vectordb = Chroma.from_documents(texts,embedding=embedddings,persist_directory='./data')

In [19]:
llm = ChatOpenAI(model="gpt-4o")

In [20]:
memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history",return_messages=True)

C:\Users\Manas\AppData\Local\Temp\ipykernel_11524\74928969.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history",return_messages=True)


In [21]:
qa=ConversationalRetrievalChain.from_llm(llm,retriever=vectordb.as_retriever(search_type="mmr",search_kwargs={"k":8}),memory=memory)

In [22]:
question = "what is download_hugging_face_embeddings function"

In [23]:
llm = ChatOpenAI(model="gpt-4o")
memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)
qa = ConversationalRetrievalChain.from_llm(llm, retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k":8}), memory=memory)
result = qa.invoke({"question": question})

In [23]:
result

{'question': 'what is download_hugging_face_embeddings function',
 'chat_history': [SystemMessage(content='', additional_kwargs={}, response_metadata={})],
 'answer': 'The `download_embeddings` function, referred to in your context, is meant to download and return embeddings from the Hugging Face model. Specifically, it uses the "sentence-transformers/all-MiniLM-L6-v2" model to obtain the embeddings. Here is the definition as per your context:\n\n```python\ndef download_embeddings():\n    """\n    Download and return the Hugging Face embeddings.\n    """\n    model_name = "sentence-transformers/all-MiniLM-L6-v2"\n    embeddings = HuggingFaceEmbeddings(model_name=model_name)\n    return embeddings\n```\n\nThis function initializes the `HuggingFaceEmbeddings` with the specified model and returns the embeddings object.'}